# Control de reglas de origen por capa (DEMO)
1. **Estático**: propuestas del agente y código de `src/silver` y `src/gold` (incluye casos negativos que deben fallar).
2. **Linaje**: `system.access.table_lineage` (lecturas reales de los pipelines; puede tener latencia).
Resultado en `workspace.estandares.resultado_control_origen`. Falla el job si hay violaciones.

In [0]:
import os, sys, io, contextlib
os.environ["CONTROL_PROPUESTAS"] = "1"
RAIZ = "/Workspace" + os.path.dirname(os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())).replace("/Workspace", "", 1)  # carpeta raíz del proyecto
os.chdir(RAIZ); sys.path.insert(0, f"{RAIZ}/tests")
import importlib, verificar_origen_capas as v
importlib.reload(v)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    codigo_estatico = v.main()
print(buf.getvalue())

In [0]:
linaje = spark.sql(open(f"{RAIZ}/tests/linaje_origen_capas.sql", encoding="utf-8").read())
display(linaje)
filas = linaje.collect()
violaciones = [r for r in filas if r.veredicto == "FAIL"]
estado_linaje = "SIN_DATOS" if not filas else ("NO_CONFORME" if violaciones else "CONFORME")

In [0]:
from pyspark.sql import Row
import datetime
spark.sql("""CREATE TABLE IF NOT EXISTS workspace.estandares.resultado_control_origen (
  fecha_control TIMESTAMP, control STRING, resultado STRING, detalle STRING)
  COMMENT 'Evidencia de auditoría: resultado de los controles de reglas de origen por capa'""")
ahora = datetime.datetime.now(datetime.timezone.utc)
spark.createDataFrame([
    Row(fecha_control=ahora, control="estatico", resultado="CONFORME" if codigo_estatico == 0 else "NO_CONFORME", detalle=buf.getvalue()[-4000:]),
    Row(fecha_control=ahora, control="linaje", resultado=estado_linaje,
        detalle="\n".join(f"{r.veredicto} | {r.destino} <- {r.origen}" for r in filas)[-4000:]),
]).write.mode("append").saveAsTable("workspace.estandares.resultado_control_origen")
print("estático:", "CONFORME" if codigo_estatico == 0 else "NO_CONFORME", "| linaje:", estado_linaje)
if codigo_estatico != 0 or violaciones:
    raise Exception("Reglas de origen por capa: NO CONFORME")